# 01 — Veri Hazırlama

**Kapsam:** Belge yönetimi ve RAG için veri altyapısı kurma.

Bu notebook üç adımı kapsar:
1. Elle yazılmış, iyi yapılandırılmış Türkçe teknik dokümantasyon şablonlarından
   (README, API dokümantasyonu, kurulum kılavuzu vb.) bir korpus oluşturma
2. Korpusu RAG için token-bazlı chunk'lara bölme
3. Chunk'lardan "taslak notlar → düzgün doküman" instruction-tuning (SFT) veri seti
   üretme (ters yönde self-instruct tekniği)

Kod, `src/data_prep/` altında modüler ve test edilebilir şekilde yazıldı; burada sadece
uçtan uca çalıştırıyoruz.

In [ ]:
# Bu hücre HER notebook'ta ayrı ayrı çalıştırılmalı: Colab'da her sekme/notebook
# genellikle kendi çalışma zamanını (VM) alır, yani /content her seferinde sıfırdanmış
# gibi başlar. Bu hücre kendi kendini onaran bir kurulum yapar:
#   1) Proje klasörü zaten varsa (aynı çalışma zamanında önceki hücre/notebook
#      tarafından kurulmuşsa) unpack adımını atlar.
#   2) Yoksa Google Drive'ı mount edip, DRIVE_ZIP_PATH'teki zip'i /content'e açar
#      (zip'in içinde 'baykar-nlp-hazirlik/' klasörü kök olarak yer almalı).
#   3) Drive'da zip de yoksa, kendi GitHub reponuzu klonlamanız için bir uyarı basar.
#   4) EN ÖNEMLİSİ: data/, models/, mlruns/ klasörlerini Drive'daki kalıcı bir
#      klasöre sembolik bağlantı (symlink) yapar. Neden gerekli: /content her
#      runtime'da sıfırlanır, yani 01. notebook'ta ürettiğiniz corpus.jsonl gibi
#      dosyalar farklı bir runtime'da (örn. 02. notebook'u açtığınızda) KAYBOLUR.
#      Bu adım olmadan her notebook'u ayrı ayrı çalıştırdığınızda önceki adımların
#      ürettiği veriyi bulamazsınız. Sembolik bağlantı sayesinde hangi runtime'da
#      olursanız olun aynı kalıcı depoyu okur/yazarsınız.
import os, sys
os.environ.setdefault("USE_TF", "0")  # transformers TensorFlow'u hic denemesin (Colab'da protobuf catismasi yasatiyor)

PROJECT_DIR = "/content/baykar-nlp-hazirlik"
DRIVE_ZIP_PATH = "/content/drive/MyDrive/baykar-nlp-hazirlik.zip"
DRIVE_DATA_DIR = "/content/drive/MyDrive/baykar-nlp-hazirlik-data"
PERSIST_DIRS = ["data/raw", "data/processed", "models", "mlruns"]  # chroma_db BILEREK haric (asagida)

try:
    from google.colab import drive
    # force_remount=True KULLANMIYORUZ: bu, zaten mount edilmişken bile her seferinde
    # yeniden yetkilendirme (izin penceresi) ister, gereksiz bekleme/kesinti yaratır.
    # drive.mount() zaten mount edilmişse kendi içinde anında geri döner; mount
    # edilmemişse (bu runtime'da ilk çalıştırma) normal şekilde izin ister — bu
    # durumda çıkan izin penceresini/bağlantısını tamamlamanız gerekir, hücreyi
    # durdurmayın.
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False  # Colab dışında (yerelde) çalışıyorsanız Drive adımları atlanır.

if not os.path.exists(PROJECT_DIR) and IN_COLAB:
    if os.path.exists(DRIVE_ZIP_PATH):
        import shutil
        shutil.unpack_archive(DRIVE_ZIP_PATH, "/content")
    else:
        print(f"UYARI: {DRIVE_ZIP_PATH} bulunamadı. Zip'i Drive'ınızın köküne "
              "yükleyin ya da kendi reponuzu klonlayın: "
              f"!git clone <repo-url> {PROJECT_DIR}")

if os.path.exists(PROJECT_DIR):
    os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)

if IN_COLAB and os.path.exists(PROJECT_DIR):
    import shutil
    os.makedirs(DRIVE_DATA_DIR, exist_ok=True)
    for _name in PERSIST_DIRS:
        _drive_path = os.path.join(DRIVE_DATA_DIR, _name)
        os.makedirs(_drive_path, exist_ok=True)
        _local_path = os.path.join(PROJECT_DIR, _name)
        os.makedirs(os.path.dirname(_local_path), exist_ok=True)  # orn. data/ klasorunu gercek dizin olarak olustur

        if os.path.islink(_local_path):
            continue  # zaten Drive'a bağlanmış

        if os.path.isdir(_local_path):
            # Zip'ten gelen boş klasörü kaldırıp yerine symlink koyuyoruz. İçinde
            # (nadiren) veri varsa önce Drive'a taşıyoruz, hiçbir şeyi kaybetmiyoruz.
            for _item in os.listdir(_local_path):
                _src = os.path.join(_local_path, _item)
                _dst = os.path.join(_drive_path, _item)
                if not os.path.exists(_dst):
                    shutil.move(_src, _dst)
            shutil.rmtree(_local_path)

        os.symlink(_drive_path, _local_path)

    print("Kalıcı veri klasörü:", DRIVE_DATA_DIR)
    print("Not: chroma_db (vektor veritabani) Drive'a BAGLANMADI -- SQLite, Drive'in")
    print("     FUSE dosya sisteminde yazma kilidini desteklemiyor ('OperationalError:")
    print("     attempt to write a readonly database'). Her yeni runtime'da RAG")
    print("     notebook'undaki (03) indeksleme hucresini tekrar calistirin -- chunks.jsonl")
    print("     zaten Drive'da oldugu icin bu hizli ve ucretsiz bir islemdir.")


## 1. Korpus oluşturma

Elle yazılmış doküman şablonlarını (README, API doc, kurulum kılavuzu vb.) yüklüyoruz.

In [ ]:
from src.data_prep.corpus_builder import build_corpus, save_corpus

corpus = build_corpus()
path = save_corpus(corpus)
print(f"{len(corpus)} şablon doküman kaydedildi -> {path}")
for doc in corpus[:3]:
    print("-", doc.title, f"({len(doc.text)} karakter)")


## 2. Chunking

Dokümanları embedding modelinin tokenizer'ıyla uyumlu, örtüşmeli parçalara bölüyoruz.

In [ ]:
from src.data_prep.chunking import chunk_corpus, save_chunks

chunks = chunk_corpus()
chunk_path = save_chunks(chunks)
print(f"{len(chunks)} chunk kaydedildi -> {chunk_path}")
print("\nÖrnek chunk:\n", chunks[0].text[:300])


## 3. Instruction (SFT) veri seti üretimi

`use_llm=True` ile temel LLM'i kullanarak her iyi-yazılmış chunk'tan, bir kullanıcının
yazacağı kaba taslak notları ürettiriyoruz (ters self-instruct). Böylece
(taslak notlar → düzgün doküman) çiftleri elde ediyoruz. GPU yoksa ya da hızlı denemek
isterseniz `use_llm=False` sezgisel (heuristic) moda düşer.

In [ ]:
from src.data_prep.instruction_dataset import build_sft_dataset, save_sft_dataset

sft_examples = build_sft_dataset(use_llm=True, max_examples=150)
sft_path = save_sft_dataset(sft_examples)
print(f"{len(sft_examples)} SFT örneği kaydedildi -> {sft_path}")
for ex in sft_examples[:2]:
    print("\nTaslak notlar (girdi):", ex.input)
    print("Düzgün doküman (hedef çıktı):", ex.output[:200], "...")
